# P18 — Prédiction de la qualité des eaux de baignade en Europe (EEA)

**Contexte métier.** Directive européenne 2006/7/CE : 21 000+ sites de baignade surveillés chaque saison et classés (excellent, bon, suffisant, mauvais). Ministères, agences sanitaires et communes doivent anticiper avant la saison les sites qui risquent de perdre leur classement — fermer un site au mauvais moment coûte du tourisme, ne pas le fermer coûte des maladies déclarées.

**Objectif de l'analyse exploratoire.** Identifier la cible (classe de qualité d'un site pour une saison) et les régresseurs disponibles avant la saison (pays, type d'eau, historique du site).

**Données.** EEA « Bathing Water Directive — Status of bathing water » (version 2025, série 1990-2024), 27 571 sites, 31 pays, export zip/xlsx sur le datahub EEA.

## 1. Téléchargement

Export officiel du datahub EEA (zip contenant le xlsx de la série 1990-2024).

In [ ]:
import os, urllib.request, zipfile

os.makedirs("data", exist_ok=True)
URL = "https://sdi.eea.europa.eu/datashare/s/J86aarkSmCMXpkc/download"
if not os.path.exists("data/bathing.zip"):
    urllib.request.urlretrieve(URL, "data/bathing.zip")
with zipfile.ZipFile("data/bathing.zip") as z:
    z.extractall("data")
print("OK", os.path.getsize("data/bathing.zip") // (1024 * 1024), "Mo")

## 2. Chargement

Le zip contient un xlsx unique (~38 Mo) : l'ouverture prend 1 à 2 minutes.

In [ ]:
import pandas as pd

df = pd.read_excel("data/eea_t_bathing-water-status_p_1990-2024_v01_r00/bw_assessment_eea_datahub_1990_2024.xlsx",
                   sheet_name="BWD_data_1990_2024", engine="openpyxl")
df.columns = [c.strip() for c in df.columns]
print(df.shape, "lignes x colonnes")
print(df.dtypes.to_string())
manq = df.isna().mean().mul(100).round(1).sort_values(ascending=False)
print("\n% manquant par colonne :")
print(manq[manq > 0].to_string())

## 3. Analyse

Sites, pays, types d'eau, distribution des classes de qualité, puis focus sur la saison 2024 et la persistance d'une année sur l'autre (baseline).

In [ ]:
print("Sites :", df["bathingWaterIdentifier"].nunique(), " pays :", df["countryCode"].nunique(),
      " saisons :", int(df["season"].min()), "-", int(df["season"].max()))
print()
print("Types d'eau :")
print(df["bathingWaterType"].value_counts(normalize=True).mul(100).round(1).to_string())
print()
print("Qualité (toutes saisons) :")
print(df["quality"].value_counts(normalize=True).mul(100).round(1).to_string())
print()
d24 = df[df["season"] == 2024]
d24 = d24[d24["quality"] != "0 - Not classified"].copy()
d24["non_excellent"] = d24["quality"] != "1 - Excellent"
print("Saison 2024 :", len(d24), "sites classés ; part non-excellent :",
      round(100 * d24["non_excellent"].mean(), 1), "%")
print()
print("Part non-excellent par type d'eau :")
print(d24.groupby("bathingWaterType")["non_excellent"].mean().mul(100).round(1).sort_values(ascending=False).to_string())
print()
prev = df[df["season"] == 2023][["bathingWaterIdentifier", "quality"]].rename(columns={"quality": "q23"})
m = d24.merge(prev, on="bathingWaterIdentifier", how="inner")
m["ne23"] = m["q23"] != "1 - Excellent"
print("Persistance (le statut 2023 prédit le statut 2024) :",
      round(100 * (m["ne23"] == m["non_excellent"]).mean(), 1), "% d'accord ;",
      "rappel sur les non-excellents 2024 :",
      round(100 * (m.loc[m["ne23"] & m["non_excellent"]].shape[0] / max(m["non_excellent"].sum(), 1)), 1), "%")

## 4. Résultats observés (exécution du 16/10/2026)

- 672 629 lignes x 13 colonnes ; 27 571 sites, 31 pays, saisons 1990-2024
- types d'eau : coastal 68,7 %, lake 26,0 %, river 4,1 %, transitional 1,2 %
- qualité toutes saisons : excellent 79,8 %, « 3 - Good or Sufficient » 7,8 % (catégorie historique), good 3,5 %, sufficient 1,0 %, poor 2,8 %, non classés 5,1 %
- saison 2024 : 21 572 sites classés ; excellent 87,2 %, good 8,9 %, sufficient 2,3 %, poor 1,7 % → part non-excellent 12,8 %
- non-excellent par type d'eau : rivière 47,0 %, transitional 18,6 %, lac 11,9 %, côte 10,4 %
- non-excellent par pays (top) : AL 84,0 %, EE 35,5 %, BE 29,7 %, PL 27,1 %, NL 26,1 %
- baseline persistance : 94,6 % d'accord entre 2023 et 2024, rappel 80,6 % sur les sites non-excellents
- valeurs manquantes : groupIdentifier 98,2 %, monitoringCalendar 77,0 %, management 76,7 %, bwProfile 47,9 %, lon/lat 2,1 %

## 5. Variable cible et régresseurs

**Cible (classification binaire) : site « non-excellent » en 2024** (`quality` != excellent), soit 12,8 % de positifs. Variante 4 classes (excellent / good / sufficient / poor) pour les étudiants avancés.

**Régresseurs retenus (disponibles avant la saison) :**
- `bathingWaterType` (rivière 47 % de non-excellent vs côte 10 % : le premier facteur)
- `countryCode` (fort effet pays : AL, EE, BE, PL...)
- `lon` / `lat` (effet géographique fin, 2,1 % de manquants)
- historique du site : classe 2023 (baseline persistance : 94,6 % d'accord), nombre de saisons surveillées, tendance
- `groupIdentifier` (sites regroupés) : manquant à 98 %, inutilisable

**Pièges :**
- « 0 - Not classified » n'est pas une qualité : l'écarter de la cible ou le traiter comme un problème à part
- la catégorie « 3 - Good or Sufficient » (7,8 % du fichier) est une nomenclature ancienne : harmoniser les étiquettes entre années
- `monitoringCalendar` et `management` sont remplis après la saison (et manquants à ~77 %) : ne pas les utiliser comme features — fuite potentielle
- la persistance est très forte : un modèle sans historique du site ne battira jamais la baseline
- les comptages par pays reflètent aussi l'effort de surveillance, pas seulement la pollution